# Processing Activity & Origins for the Frontend

This notebook turns the exploratory analysis in `explore_activity_and_origins.ipynb` into **production JSON** the frontend can load directly. Two graphics are being built:

1. **Line graphs** — Main 2026 vs. the 2025 benchmark vs. the pre-2026 period, day by day, for daily activity.
2. **Proportional-symbol maps** — mean visitor share by home region across match days, at both the metro scale and the Canada/US-wide scale.

**Scope**: only 4 sources are produced for either graphic - `toronto_csd`, `toronto_merged_2km`, `vancouver_csd`, `vancouver_merged_2km` (no `merged_5km`, matching what the exploration notebook's maps used).

**Output format decisions** (confirmed before writing any code):
- JSON keys use **camelCase** (`periodDayNumber`, `activityMain2026`, `isMatchDay`, ...).
- Dates are **ISO strings** (`"2026-06-01"`), directly parseable by JS `new Date(...)`.
- `periodDayNumber` is **1-indexed** (day 1 = first day of the period).
- Map JSON entries are **centroid point + share only** (`region`, `lat`, `lon`, `pct`) - no boundary polygons; the frontend supplies its own basemap.
- Missing values (e.g. the 2025 benchmark running short) are written as JSON **`null`**, never `NaN` - `json.dump(..., allow_nan=False)` is used everywhere as a hard guardrail so a stray `NaN` raises immediately instead of silently producing invalid JSON a JS `JSON.parse` would choke on.

**A known data quality issue**: `origin_by_region_just_num_rings.xlsx`'s `NAWidePct` column has a bug (documented in the exploration notebook) where minor regions all share one broadcast residual value instead of their own share. The fix (recomputing from `AdjHomeDevices`) lives in its **own cell** (Section 2b) so it can be deleted cleanly once the upstream data is corrected - no need to touch anything else in the notebook when that happens.

**Output location**: `data/frontend/` - structured so the whole folder can be copied straight into the frontend project:
```
data/frontend/
  activity/
    toronto_csd.json
    toronto_2km.json
    vancouver_csd.json
    vancouver_2km.json
  origins/
    metro/
      toronto_csd.json, toronto_2km.json, vancouver_csd.json, vancouver_2km.json
    north_america/
      toronto_csd.json, toronto_2km.json, vancouver_csd.json, vancouver_2km.json
```


In [10]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import geopandas as gpd
from tqdm.notebook import tqdm

pd.options.display.float_format = "{:,.4f}".format


In [11]:
ACTIVITY_PATH = "../../data/match-activity/daily_final_just_num.xlsx"
ORIGIN_PATH = "../../data/match-activity/origin_by_region_just_num_rings.xlsx"
GEOMS_DIR = "../../data/final_geoms"

# The 4 sources produced for both graphics - (city, level) -> output file slug
SOURCES = [
    ("toronto", "csd"), ("toronto", "merged_2km"),
    ("vancouver", "csd"), ("vancouver", "merged_2km"),
]
FILE_SLUG = {
    ("toronto", "csd"): "toronto_csd", ("toronto", "merged_2km"): "toronto_2km",
    ("vancouver", "csd"): "vancouver_csd", ("vancouver", "merged_2km"): "vancouver_2km",
}

# Output folder - copy this whole tree into the frontend project
OUTPUT_DIR = Path("../../data/frontend")
ACTIVITY_DIR = OUTPUT_DIR / "activity"
ORIGINS_METRO_DIR = OUTPUT_DIR / "origins" / "metro"
ORIGINS_NA_DIR = OUTPUT_DIR / "origins" / "north_america"
for d in [ACTIVITY_DIR, ORIGINS_METRO_DIR, ORIGINS_NA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Time periods (dates as they actually appear in daily_final_just_num.xlsx - verified in the exploration notebook)
BENCHMARK_2025 = ("2025-06-01", "2025-07-10")
PRE_2026 = ("2026-04-22", "2026-05-31")
MAIN_2026 = ("2026-06-01", "2026-07-10")

# FIFA World Cup 2026 match days per host city, within MAIN_2026.
# Includes later-round dates (2026-07-02, 2026-07-07) found in the origin file beyond the original group-stage list.
MATCH_DAYS = {
    "toronto": pd.to_datetime(["2026-06-12", "2026-06-17", "2026-06-20", "2026-06-23", "2026-06-26", "2026-07-02"]),
    "vancouver": pd.to_datetime(["2026-06-13", "2026-06-18", "2026-06-21", "2026-06-24", "2026-06-26",
                                  "2026-07-02", "2026-07-07"]),
}

# --- Geometry sources for the map JSONs (data/final_geoms) - same setup as the exploration notebook ---
CSD_SHAPEFILES = {
    "toronto": f"{GEOMS_DIR}/csd_gtha/csds_metro_gtha.shp",
    "vancouver": f"{GEOMS_DIR}/csd_vancouver/csds_metro_van.shp",
}
CSD_NAME_TO_REGION = {"Toronto": "Toronto CSD", "Vancouver": "Vancouver CSD"}

CMA_SHAPEFILES = {
    "toronto": f"{GEOMS_DIR}/cma_gtha/toronto_hamilton_cma.shp",
    "vancouver": f"{GEOMS_DIR}/cma_vancouver/vancouver_cma.shp",
}
CMA_REGION_NAME = {"toronto": "Toronto Hamilton CMA", "vancouver": "Vancouver CMA"}
REST_OF_PROVINCE_REGION_NAME = {"toronto": "Ontario - GTHA", "vancouver": "BC - Metro Van"}
HOME_PROVINCE = {"toronto": "Ontario", "vancouver": "British Columbia"}

PROVINCE_SHAPEFILES = {
    "Ontario": "on", "British Columbia": "bc", "Alberta": "ab", "Manitoba": "mb", "Saskatchewan": "sk",
    "Quebec": "qb", "Nova Scotia": "ns", "New Brunswick": "nb", "Newfoundland and Labrador": "nl",
    "Prince Edward Island": "pe", "Yukon": "yk", "Northwest Territories": "nt", "Nunavut": "nu",
}

# No Canadian boundary file covers the US as a whole - approximate center of the contiguous US
US_COORD = (39.8, -98.6)

# Rounding applied just before writing JSON - keeps files compact without losing meaningful precision
ROUND_ACTIVITY = 6   # normed_devices values are ~0.005-0.013
ROUND_PCT = 3         # percent-difference and share values, reported as percent points
ROUND_COORD = 5       # ~1m precision for lat/lon


## 1. Load source data

Same stacking approach as the exploration notebook: both source files are one-sheet-per-`{city}_{geometry}`, so we read every sheet and filter down to just the 4 sources this notebook produces output for.


In [12]:
xl = pd.ExcelFile(ACTIVITY_PATH)
frames = []
for city, level in tqdm(SOURCES, desc="activity sources"):
    d = xl.parse(f"{city}_{level}")
    d["city"], d["level"] = city, level
    frames.append(d)

activity = pd.concat(frames, ignore_index=True)
activity["DATE"] = pd.to_datetime(activity["DATE"], format="%Y%m%d")
assert not activity.duplicated(subset=["DATE", "GEOMETRYLEVEL"]).any()

print("Loaded activity rows:", len(activity), "| sources:", sorted(activity['GEOMETRYLEVEL'].unique()))
activity.head(3)


activity sources:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded activity rows: 480 | sources: ['toronto_csd', 'toronto_merged_2km', 'vancouver_csd', 'vancouver_merged_2km']


,DATE,UNIQUESTOPS,GEOMETRYLEVEL,denom_for_geom_lvl,normed_devices,city,level
0,2025-06-01,259547,toronto_csd,33195602,0.0078,toronto,csd
1,2025-06-02,255354,toronto_csd,33195602,0.0077,toronto,csd
2,2025-06-03,251762,toronto_csd,33195602,0.0076,toronto,csd


In [13]:
xl_o = pd.ExcelFile(ORIGIN_PATH)
frames = []
for city, level in tqdm(SOURCES, desc="origin sources"):
    sheet = f"{city}_{level}"
    d = xl_o.parse(sheet)
    d["city"], d["level"], d["source"] = city, level, sheet
    frames.append(d)

origin = pd.concat(frames, ignore_index=True)
origin["DATE"] = pd.to_datetime(origin["DATE"], format="%Y%m%d")

print("Loaded origin rows:", len(origin), "| sources:", sorted(origin["source"].unique()))
origin.head(3)


origin sources:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded origin rows: 995 | sources: ['toronto_csd', 'toronto_merged_2km', 'vancouver_csd', 'vancouver_merged_2km']


,DATE,GEOMETRYLEVEL,REGION,HomeDevices,DenomGeometry,GameDayDenom,AvgMayDevices,ScaleFactor,AdjHomeDevices,WithinGTHAPct,NAWidePct,city,level,source,WithinMetroVanPct
0,2026-06-12,toronto_csd,Ontario,223935,"""on""",1470396,"1,190,547.5000",0.8097,"181,315.2745",NaN,NaN,toronto,csd,toronto_csd,NaN
1,2026-06-12,toronto_csd,Toronto Hamilton CMA,212370,toronto_hamilton_cma,828155,"658,389.0000",0.7950,"168,835.6309",NaN,0.9260,toronto,csd,toronto_csd,NaN
2,2026-06-12,toronto_csd,Toronto CSD,190566,toronto_csd,339453,"290,820.2500",0.8567,"163,263.9917",0.9094,NaN,toronto,csd,toronto_csd,NaN


## 2b. Data-quality fix (delete once the upstream data is corrected)

`NAWidePct` in `origin_by_region_just_num_rings.xlsx` has a real bug: every "minor" region (small provinces, US) on a given date shares one identical broadcast residual value instead of its own share (see `explore_activity_and_origins.ipynb` section 2a for the full demonstration). `WithinGTHAPct`/`WithinMetroVanPct` don't have this problem.

This cell recomputes both percentage columns directly from `AdjHomeDevices`, which is reliable at every level. **This entire cell can be deleted once the source file is fixed upstream** - nothing downstream depends on how the fix is applied, only on `metro_pct`/`na_pct` existing and being correct.


In [14]:
# --- DELETE THIS CELL once origin_by_region_just_num_rings.xlsx's NAWidePct bug is fixed upstream ---
origin["metro_pct_orig"] = origin["WithinGTHAPct"].combine_first(origin["WithinMetroVanPct"])

def recompute_share(df, membership_col, new_col):
    in_group = df[membership_col].notna()
    totals = df.loc[in_group].groupby(["source", "DATE"])["AdjHomeDevices"].transform("sum")
    df[new_col] = np.nan
    df.loc[in_group, new_col] = df.loc[in_group, "AdjHomeDevices"] / totals
    return df

origin = recompute_share(origin, "metro_pct_orig", "metro_pct")
origin = recompute_share(origin, "NAWidePct", "na_pct")

sums = pd.concat([
    origin.groupby(["source", "DATE"])["metro_pct"].sum(min_count=1).dropna(),
    origin.groupby(["source", "DATE"])["na_pct"].sum(min_count=1).dropna(),
])
assert np.allclose(sums, 1.0), "recomputed shares should sum to 1 per source/date"
print("metro_pct / na_pct recomputed from AdjHomeDevices - all groups sum to 1.0")
# --- END DELETE ---


metro_pct / na_pct recomputed from AdjHomeDevices - all groups sum to 1.0


## 3. Build the activity line-graph JSON

One list per source, one dict per **period day** (1-indexed, day 1 = the first day of each period). Each dict aligns the same day-offset across all three periods - `2025 benchmark` (June 2025, 30 days), `pre-2026` (Apr 22 - May 31 2026, 40 days), and `main 2026` (Jun 1 - Jul 10 2026, 40 days) - so day 15 means "the 15th day into whichever period," not a shared calendar date. The list runs to the longest period (40 days); since the 2025 benchmark is only 30 days, days 31-40 have no corresponding 2025 date or value at all, so `date2025`/`activity2025`/`pctVs2025` are `null` there - not a data gap to be alarmed by, just the benchmark being a shorter, fixed calendar month while the other two periods are ~40-day windows.

Only `normed_devices` (the normalized activity metric) is used, per the brief. `pctVs2025`/`pctVsPre2026` are `100 * (mainValue / otherValue - 1)` - positive means Main 2026 is higher.


In [15]:
def lookup_activity(series, date, start, end):
    if date < start or date > end:
        return None
    val = series.get(date)
    if val is None or pd.isna(val):
        return None
    return round(float(val), ROUND_ACTIVITY)

def pct_diff(main_val, other_val):
    if main_val is None or other_val is None or other_val == 0:
        return None
    return round(100 * (main_val / other_val - 1), ROUND_PCT)

def build_activity_records(city, level):
    series = activity[(activity["city"] == city) & (activity["level"] == level)].set_index("DATE")["normed_devices"]

    bench_start, bench_end = pd.to_datetime(BENCHMARK_2025[0]), pd.to_datetime(BENCHMARK_2025[1])
    pre_start, pre_end = pd.to_datetime(PRE_2026[0]), pd.to_datetime(PRE_2026[1])
    main_start, main_end = pd.to_datetime(MAIN_2026[0]), pd.to_datetime(MAIN_2026[1])

    n_days = max((bench_end - bench_start).days, (pre_end - pre_start).days, (main_end - main_start).days) + 1

    records = []
    for i in range(n_days):
        bench_date = bench_start + pd.Timedelta(days=i)
        pre_date = pre_start + pd.Timedelta(days=i)
        main_date = main_start + pd.Timedelta(days=i)

        in_bench = bench_start <= bench_date <= bench_end
        in_main = main_start <= main_date <= main_end

        bench_val = lookup_activity(series, bench_date, bench_start, bench_end)
        pre_val = lookup_activity(series, pre_date, pre_start, pre_end)
        main_val = lookup_activity(series, main_date, main_start, main_end)

        records.append({
            "periodDayNumber": i + 1,
            "date2025": bench_date.date().isoformat() if in_bench else None,
            "datePre2026": pre_date.date().isoformat(),
            "dateMain2026": main_date.date().isoformat(),
            "activity2025": bench_val,
            "activityPre2026": pre_val,
            "activityMain2026": main_val,
            "pctVs2025": pct_diff(main_val, bench_val),
            "pctVsPre2026": pct_diff(main_val, pre_val),
            "isMatchDay": bool(in_main and main_date in MATCH_DAYS[city]),
        })
    return records

for city, level in tqdm(SOURCES, desc="activity JSON"):
    records = build_activity_records(city, level)
    out_path = ACTIVITY_DIR / f"{FILE_SLUG[(city, level)]}.json"
    with open(out_path, "w") as f:
        json.dump(records, f, allow_nan=False, indent=2)
    print(f"{out_path} - {len(records)} days")

records


activity JSON:   0%|          | 0/4 [00:00<?, ?it/s]

../../data/frontend/activity/toronto_csd.json - 40 days
../../data/frontend/activity/toronto_2km.json - 40 days
../../data/frontend/activity/vancouver_csd.json - 40 days
../../data/frontend/activity/vancouver_2km.json - 40 days


[{'periodDayNumber': 1,
  'date2025': '2025-06-01',
  'datePre2026': '2026-04-22',
  'dateMain2026': '2026-06-01',
  'activity2025': 0.007962,
  'activityPre2026': 0.00743,
  'activityMain2026': 0.006386,
  'pctVs2025': -19.794,
  'pctVsPre2026': -14.051,
  'isMatchDay': False},
 {'periodDayNumber': 2,
  'date2025': '2025-06-02',
  'datePre2026': '2026-04-23',
  'dateMain2026': '2026-06-02',
  'activity2025': 0.008379,
  'activityPre2026': 0.00703,
  'activityMain2026': 0.006469,
  'pctVs2025': -22.795,
  'pctVsPre2026': -7.98,
  'isMatchDay': False},
 {'periodDayNumber': 3,
  'date2025': '2025-06-03',
  'datePre2026': '2026-04-24',
  'dateMain2026': '2026-06-03',
  'activity2025': 0.008281,
  'activityPre2026': 0.007535,
  'activityMain2026': 0.008476,
  'pctVs2025': 2.355,
  'pctVsPre2026': 12.488,
  'isMatchDay': False},
 {'periodDayNumber': 4,
  'date2025': '2025-06-04',
  'datePre2026': '2026-04-25',
  'dateMain2026': '2026-06-04',
  'activity2025': 0.008558,
  'activityPre2026': 

## 4. Build the proportional-symbol map JSON

Region locations come from the real Statistics Canada boundary files in `data/final_geoms/` (same approach as the exploration notebook): each region's true polygon centroid, with "rest of province" computed from the actual province-minus-CMA geometric difference rather than a hand-picked point.

Each output file is a list of `{region, lat, lon, pct}` - the **mean share across all match days** for that region, sorted largest-first. Two scales, one file each per source:
- **`origins/metro/`** - the municipality-level breakdown within the home CMA (`metro_pct`).
- **`origins/north_america/`** - the home CMA, the rest of the home province, other provinces, and the US (`na_pct`).


In [16]:
def polygon_centroid_latlon(geom_series):
    """Centroid (lat, lon) of a (possibly multi-row) GeoSeries, computed in its own projected CRS."""
    # union_all() (geopandas >=0.14/1.0) vs. unary_union (older geopandas) - support both kernels/environments.
    merged = geom_series.union_all() if hasattr(geom_series, "union_all") else geom_series.unary_union
    pt = gpd.GeoSeries([merged.centroid], crs=geom_series.crs).to_crs(4326).iloc[0]
    return (pt.y, pt.x)

REGION_COORDS = {}

# CSD (municipality) centroids - dissolve by name first (a couple of municipalities, e.g. Langley,
# North Vancouver, are split into 2 polygons sharing one name)
for city, path in tqdm(CSD_SHAPEFILES.items(), desc="csd geometries"):
    gdf = gpd.read_file(path)
    for name, group in gdf.groupby("CSDNAME"):
        region = CSD_NAME_TO_REGION.get(name, name)
        REGION_COORDS[region] = polygon_centroid_latlon(group.geometry)

# CMA (metro) centroids
cma_geoms = {}
for city, path in CMA_SHAPEFILES.items():
    gdf = gpd.read_file(path)
    cma_geoms[city] = gdf.geometry.iloc[0]
    REGION_COORDS[CMA_REGION_NAME[city]] = polygon_centroid_latlon(gdf.geometry)

# Province/territory centroids
province_geoms = {}
for prov_name, abbr in tqdm(PROVINCE_SHAPEFILES.items(), desc="province geometries"):
    gdf = gpd.read_file(f"{GEOMS_DIR}/provinces_territories/{abbr}.shp")
    province_geoms[prov_name] = gdf
    REGION_COORDS[prov_name] = polygon_centroid_latlon(gdf.geometry)

# "Rest of the home province" = province polygon minus its own CMA polygon, via real geometric difference
for city in ["toronto", "vancouver"]:
    prov_gdf = province_geoms[HOME_PROVINCE[city]]
    rest_geom = prov_gdf.geometry.iloc[0].difference(cma_geoms[city])
    pt = gpd.GeoSeries([rest_geom.centroid], crs=prov_gdf.crs).to_crs(4326).iloc[0]
    REGION_COORDS[REST_OF_PROVINCE_REGION_NAME[city]] = (pt.y, pt.x)

REGION_COORDS["US"] = US_COORD

print(f"Built real-geometry coordinates for {len(REGION_COORDS)} regions")


csd geometries:   0%|          | 0/2 [00:00<?, ?it/s]

province geometries:   0%|          | 0/13 [00:00<?, ?it/s]

Built real-geometry coordinates for 80 regions


In [17]:
SCALE_OUTPUT_DIRS = {"metro_pct": ORIGINS_METRO_DIR, "na_pct": ORIGINS_NA_DIR}

mean_pct = (
    origin.melt(id_vars=["source", "city", "REGION"], value_vars=["metro_pct", "na_pct"],
                var_name="scale", value_name="pct")
    .dropna(subset=["pct"])
    .groupby(["source", "scale", "REGION"])["pct"].mean()
    .reset_index()
)

missing = set(mean_pct["REGION"]) - set(REGION_COORDS)
assert not missing, f"Add coordinates for: {missing}"

for scale, out_dir in SCALE_OUTPUT_DIRS.items():
    for city, level in tqdm(SOURCES, desc=f"{scale} JSON"):
        source = f"{city}_{level}"
        d = mean_pct[(mean_pct["source"] == source) & (mean_pct["scale"] == scale)].sort_values("pct", ascending=False)

        records = [
            {
                "region": row.REGION,
                "lat": round(REGION_COORDS[row.REGION][0], ROUND_COORD),
                "lon": round(REGION_COORDS[row.REGION][1], ROUND_COORD),
                "pct": round(float(row.pct) * 100, ROUND_PCT),  # stored as a proportion upstream - convert to a percentage
            }
            for row in d.itertuples()
        ]

        out_path = out_dir / f"{FILE_SLUG[(city, level)]}.json"
        with open(out_path, "w") as f:
            json.dump(records, f, allow_nan=False, indent=2)
        print(f"{out_path} - {len(records)} regions")

records


metro_pct JSON:   0%|          | 0/4 [00:00<?, ?it/s]

../../data/frontend/origins/metro/toronto_csd.json - 26 regions
../../data/frontend/origins/metro/toronto_2km.json - 26 regions
../../data/frontend/origins/metro/vancouver_csd.json - 26 regions
../../data/frontend/origins/metro/vancouver_2km.json - 25 regions


na_pct JSON:   0%|          | 0/4 [00:00<?, ?it/s]

../../data/frontend/origins/north_america/toronto_csd.json - 15 regions
../../data/frontend/origins/north_america/toronto_2km.json - 14 regions
../../data/frontend/origins/north_america/vancouver_csd.json - 14 regions
../../data/frontend/origins/north_america/vancouver_2km.json - 14 regions


[{'region': 'Vancouver CMA',
  'lat': 49.28232,
  'lon': -122.83573,
  'pct': 88.199},
 {'region': 'BC - Metro Van', 'lat': 54.32542, 'lon': -124.71012, 'pct': 6.41},
 {'region': 'US', 'lat': 39.8, 'lon': -98.6, 'pct': 2.436},
 {'region': 'Alberta', 'lat': 54.93595, 'lon': -114.38676, 'pct': 1.709},
 {'region': 'Ontario', 'lat': 50.03065, 'lon': -85.48147, 'pct': 0.825},
 {'region': 'Manitoba', 'lat': 54.68875, 'lon': -97.50199, 'pct': 0.25},
 {'region': 'Saskatchewan', 'lat': 54.13282, 'lon': -105.87179, 'pct': 0.087},
 {'region': 'Yukon', 'lat': 63.47412, 'lon': -135.18791, 'pct': 0.03},
 {'region': 'Nova Scotia', 'lat': 45.15543, 'lon': -63.31958, 'pct': 0.018},
 {'region': 'Newfoundland and Labrador',
  'lat': 52.58558,
  'lon': -59.48557,
  'pct': 0.016},
 {'region': 'Northwest Territories',
  'lat': 71.69373,
  'lon': -121.55077,
  'pct': 0.015},
 {'region': 'New Brunswick', 'lat': 46.5877, 'lon': -66.311, 'pct': 0.013},
 {'region': 'Prince Edward Island',
  'lat': 46.34748,
  'l

## 5. Verify the output

Sanity-check every written file: valid JSON (round-trips through `json.load`), no `NaN`/`Infinity` tokens, and the expected row counts.


In [18]:
json_files = sorted(OUTPUT_DIR.rglob("*.json"))

rows = []
for path in tqdm(json_files, desc="verifying JSON"):
    raw = path.read_text()
    assert "NaN" not in raw and "Infinity" not in raw, f"{path} contains a non-JSON-safe token"
    data = json.loads(raw)  # raises if invalid JSON
    rows.append({
        "file": str(path.relative_to(OUTPUT_DIR)),
        "records": len(data),
        "size_kb": round(len(raw) / 1024, 1),
    })

print(f"{len(json_files)} files, all valid JSON, no NaN/Infinity tokens")
pd.DataFrame(rows)


verifying JSON:   0%|          | 0/12 [00:00<?, ?it/s]

12 files, all valid JSON, no NaN/Infinity tokens


,file,records,size_kb
0,activity/toronto_2km.json,40,12.0000
1,activity/toronto_csd.json,40,12.0000
2,activity/vancouver_2km.json,40,12.0000
3,activity/vancouver_csd.json,40,12.0000
4,origins/metro/toronto_2km.json,26,2.5000
5,origins/metro/toronto_csd.json,26,2.5000
6,origins/metro/vancouver_2km.json,25,2.4000
7,origins/metro/vancouver_csd.json,26,2.5000
8,origins/north_america/toronto_2km.json,14,1.4000
9,origins/north_america/toronto_csd.json,15,1.5000
